In [6]:
import pandas as pd

# load dataset
df = pd.read_csv('C:\\Users\\ridaa\\friends-nlp-project\\data\\archive (4)\\Friends.csv')

# check structure
print(df.head())
print(df.columns)
print(df.shape)


                                                Text  Speaker  \
0  Originally written by Marta Kauffman and David...      NaN   
1                          Transcribed by guineapig.      NaN   
2   CENTRAL PERK. (ALL PRESENT EXCEPT RACHEL AND ...  SCENE 1   
3   There's nothing to tell! He's just some guy I...   MONICA   
4   C'mon, you're going out with the guy! There's...     JOEY   

                                             Episode     Season     Show  
0  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
1  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
2  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
3  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
4  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
Index(['Text', 'Speaker', 'Episode', 'Season', 'Show'], dtype='object')
(69974, 5)


In [7]:
import pandas as pd

df = pd.read_csv('C:\\Users\\ridaa\\friends-nlp-project\\data\\archive (4)\\Friends.csv')

# 1. Drop rows where speaker is missing
df = df.dropna(subset=['Speaker'])

# 2. Remove rows where speaker is not a character (like SCENE)
df = df[~df['Speaker'].str.contains('SCENE', case=False)]

# 3. Strip whitespace
df['Speaker'] = df['Speaker'].str.strip()
df['Text'] = df['Text'].str.strip()

# 4. Remove very short lines (like noise)
df = df[df['Text'].str.len() > 5]

# 5. Keep only main characters (IMPORTANT for better model)
main_chars = ['JOEY', 'ROSS', 'CHANDLER', 'MONICA', 'RACHEL', 'PHOEBE']
df = df[df['Speaker'].isin(main_chars)]

# Reset index
df = df.reset_index(drop=True)

print(df.head())
print(df['Speaker'].value_counts())
print(df.shape)

                                                Text   Speaker  \
0  There's nothing to tell! He's just some guy I ...    MONICA   
1  C'mon, you're going out with the guy! There's ...      JOEY   
2    So does he have a hump? A hump and a hairpiece?  CHANDLER   
3                           Wait, does he eat chalk?    PHOEBE   
4  Just, 'cause, I don't want her to go through w...    PHOEBE   

                                             Episode     Season     Show  
0  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
1  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
2  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
3  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
4  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
Speaker
ROSS        1315
CHANDLER    1048
MONICA      1033
JOEY        1005
RACHEL       930
PHOEBE       798
Name: count, dtype: int64
(6129, 5)


In [8]:
import re

# Improved catchphrases with flexible patterns
catchphrases = {
    "JOEY": [
        r"how\s+you\s+doin['g]?"
    ],
    "ROSS": [
        r"we\s+were\s+on\s+a\s+break"
    ],
    "CHANDLER": [
        r"could\s+i\s+be",
        r"could\s+that\s+be"
    ],
    "PHOEBE": [
        r"smelly\s+cat"
    ],
}

def preprocess(text):
    # lowercase + remove punctuation
    text = text.lower()
    text = re.sub(r"[^\w\s']", "", text)
    return text

def detect_catchphrase(text):
    clean_text = preprocess(text)

    for character, patterns in catchphrases.items():
        for pattern in patterns:
            if re.search(pattern, clean_text):
                return character, pattern

    return None, None


# Test
samples = [
    "How you doin?",
    "HOW YOU DOIN!!!",
    "We were on a break!",
    "we WERE on a break...",
    "Could I BE any more tired?",
    "Could that BE more obvious?",
    "I love smelly cat song",
]

for s in samples:
    print(s, "->", detect_catchphrase(s))

How you doin? -> ('JOEY', "how\\s+you\\s+doin['g]?")
HOW YOU DOIN!!! -> ('JOEY', "how\\s+you\\s+doin['g]?")
We were on a break! -> ('ROSS', 'we\\s+were\\s+on\\s+a\\s+break')
we WERE on a break... -> ('ROSS', 'we\\s+were\\s+on\\s+a\\s+break')
Could I BE any more tired? -> ('CHANDLER', 'could\\s+i\\s+be')
Could that BE more obvious? -> ('CHANDLER', 'could\\s+that\\s+be')
I love smelly cat song -> ('PHOEBE', 'smelly\\s+cat')


In [12]:
import random
import re

def friends_translator(text, character):
    text = text.strip()

    if not text:
        return "Say something!"

    lower_text = text.lower()

    if character == "JOEY":
        templates = [
            f"{text}... How you doin'?",
            f"{text}? How YOU doin'?",
        ]
        return random.choice(templates)

    elif character == "CHANDLER":
        word = lower_text.replace("i'm", "").replace("i am", "").strip()

        if not word:
            word = "this"

        templates = [
            f"Could I BE any more {word}?",
            f"Oh wow, could that BE any more {word}?",
            f"Could this situation BE any more {word}?"
        ]
        return random.choice(templates)

    elif character == "ROSS":
        if re.search(r"we\s+were\s+on\s+a\s+break", lower_text):
            return "WE WERE ON A BREAK!"

        templates = [
            f"{text}!!!",
            f"I mean, {text}...",
            f"{text}! This is unbelievable!",
        ]
        return random.choice(templates)

    elif character == "PHOEBE":
        # FIXED indentation + logic
        if "my eyes" in lower_text and lower_text.count("my eyes") >= 2:
            return "MY EYES! MY EYES!"

        templates = [
            f"{text}... but like in a weird, cosmic way.",
            f"{text}... ooooh that reminds me of a song!",
            f"This is brand new information!",
            f"{text}... but what if it's actually a spirit?"
        ]
        return random.choice(templates)

    elif character == "MONICA":
        templates = [
            f"{text}. Okay, we need a plan. RIGHT NOW.",
            f"{text}! This is completely out of control!",
            f"{text}. I’ll fix this.",
            f"{text}! I KNOW!"
        ]
        return random.choice(templates)

    elif character == "RACHEL":
        templates = [
            f"{text}?! No way!",
            f"Oh my God, {text}!",
            f"{text}... I can’t even.",
            f"{text}! Are you serious?"
        ]
        return random.choice(templates)

    else:
        return text

In [13]:
print(friends_translator("I'm tired", "CHANDLER"))
print(friends_translator("I'm hungry", "JOEY"))
print(friends_translator("This is unfair", "ROSS"))

Oh wow, could that BE any more tired?
I'm hungry... How you doin'?
This is unfair!!!


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pickle

# 1. Fit vectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['Text'])

# 2. Train model
model = LogisticRegression()
model.fit(X, df['Speaker'])  # or your label column

# 3. Save
with open('../models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

with open('../models/model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Saved successfully")

Saved successfully
